# AI Post Studio — Google Colab (v10 updated, references + biodata)

Aapke diye hue v10 app ko update kiya gaya hai: natural Indian matrimonial portraits, full-face framing, optional bindi (sindoor nahi), reusable reference/avoid notes, selected image se biodata, an optional **Bold** look, and a clearer restart flow.

**Run:** select *Runtime → Change runtime type → T4 GPU*, then run the cells from top to bottom. Approve Google Drive access and wait for the final **Start / Restart** cell to print a clickable `gradio.live` link.

**If the link does not appear:** use this updated notebook’s Start / Restart cell. It waits for the public link, shows the app log if tunnel creation fails, and starts Python without buffered output. If Colab says “No interface is running,” reconnect the runtime and run Start / Restart again; rerun Setup only if the app files or model are missing.

Images tab ki gallery mein kisi saved photo par click karke Biodata tab mein **Selected photo se biodata banao** chunein; random details usi photo par lagengi. Settings mein **Reference se kya follow karein** aur **Kin cheezon se bachein** save karke future image aur biodata prompts guide karein. Chhoti bindi allowed hai, maang mein sindoor nahi. Notebook app code, tests, aur Hindi fonts ke saath self-contained hai; GitHub upload ki zaroorat nahi. Photos, posts, biodata, settings, aur model `MyDrive/AI_Post_Studio` mein save hote hain.

In [ ]:
#@title 1. Settings
TEXT_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"  #@param ["Qwen/Qwen2.5-1.5B-Instruct", "Qwen/Qwen2.5-3B-Instruct", "Off (sirf template, sabse fast)"]
PHOTO_MODEL = "RealVisXL V5 (sabse real)"  #@param ["RealVisXL V5 (sabse real)", "Juggernaut XL v9 (achhe kapde/light)"]
MODELS = {
    "RealVisXL V5 (sabse real)": "https://huggingface.co/SG161222/RealVisXL_V5.0/resolve/main/RealVisXL_V5.0_fp16.safetensors",
    "Juggernaut XL v9 (achhe kapde/light)": "https://huggingface.co/RunDiffusion/Juggernaut-XL-v9/resolve/main/Juggernaut-XL_v9_RunDiffusionPhoto_v2.safetensors",
}
CHECKPOINT_URL = MODELS[PHOTO_MODEL]
BASE = "/content/drive/MyDrive/AI_Post_Studio"


In [ ]:
#@title 2. Google Drive mount
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs(BASE, exist_ok=True)
print('Drive ready:', BASE)


In [ ]:
#@title 3. One-click setup (updated app included; first run downloads models)
import os, subprocess, shutil, sys, json, time, base64, io, zipfile
from pathlib import Path

PAYLOAD_B64 = "..."
repo_dir = "/content/aps"
Path(repo_dir).mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(io.BytesIO(base64.b64decode(PAYLOAD_B64))) as archive:
    archive.extractall(repo_dir)
if not os.path.isfile(f"{repo_dir}/app/main.py"):
    raise RuntimeError("Embedded app files nahi mile; notebook file ko dobara upload karein.")

def sh(command, must=True):
    print('$', command)
    result = subprocess.run(command, shell=True)
    if result.returncode and must:
        raise RuntimeError(f"Setup command failed ({result.returncode}): {command}")
    return result.returncode == 0

def clone_repo(url, path, must=True):
    if os.path.isfile(f"{path}/main.py"):
        return True
    shutil.rmtree(path, ignore_errors=True)
    os.makedirs(os.path.dirname(path), exist_ok=True)
    return sh(f'git clone -q --depth 1 "{url}" "{path}"', must)

sh('apt-get -qq update && apt-get -qq install -y libraqm0 libfribidi0 aria2 git > /dev/null')
# Colab's temporary disk keeps ComfyUI code and its runtime output out of Drive.
comfy = '/content/ComfyUI'
if not clone_repo('https://github.com/comfyanonymous/ComfyUI', comfy, False):
    raise RuntimeError('ComfyUI download fail hua. Runtime restart karke setup dobara chalayein.')
sh(f'{sys.executable} -m pip install -q -r "{comfy}/requirements.txt"')
sh(f'{sys.executable} -m pip install -q -r "{repo_dir}/requirements.txt"')

# Store large reusable models and the user's work in Drive; link model folders to local ComfyUI.
model_root = Path(BASE) / 'models'
for folder in ('checkpoints', 'upscale_models', 'vae', 'ultralytics/bbox'):
    (model_root / folder).mkdir(parents=True, exist_ok=True)
for folder in ('checkpoints', 'upscale_models', 'vae', 'ultralytics'):
    local = Path(comfy) / 'models' / folder
    saved = model_root / folder
    if local.is_symlink():
        local.unlink()
    elif local.exists() and local.is_dir():
        shutil.rmtree(local)
    local.parent.mkdir(parents=True, exist_ok=True)
    if not saved.exists(): saved.mkdir(parents=True, exist_ok=True)
    os.symlink(saved, local, target_is_directory=True)

# Resume partial model downloads. Do not treat a leftover partial .safetensors as complete.
checkpoint = model_root / 'checkpoints'
model_name = CHECKPOINT_URL.split('/')[-1].split('?')[0]
model_path = checkpoint / model_name
aria_control = Path(str(model_path) + '.aria2')
if not model_path.is_file() or aria_control.exists() or model_path.stat().st_size < 2_500_000_000:
    print(f'Model: {model_name} — downloading/resuming in Google Drive. Please let this cell finish.')
    sh(f'aria2c --console-log-level=notice -c -x 8 -s 8 --file-allocation=none -d "{checkpoint}" -o "{model_name}" "{CHECKPOINT_URL}"')
if not model_path.is_file() or model_path.stat().st_size < 2_500_000_000 or aria_control.exists():
    raise RuntimeError('Photo model download adhoora hai. Setup cell dobara chalayein; aria2c partial download resume karega. App abhi start nahi hua.')
print(f'Photo model ready: {model_name} ({model_path.stat().st_size / 1e9:.2f} GB)')

# Quality assets: the main checkpoint is required; detailer assets stay optional.
upscaler = model_root / 'upscale_models' / '4x-UltraSharp.pth'
if not upscaler.is_file():
    sh(f'wget -q -O "{upscaler}" https://huggingface.co/lokCX/4x-Ultrasharp/resolve/main/4x-UltraSharp.pth', False)
vae = model_root / 'vae' / 'sdxl_vae_fp16_fix.safetensors'
if not vae.is_file():
    sh(f'wget -q -O "{vae}" https://huggingface.co/madebyollin/sdxl-vae-fp16-fix/resolve/main/sdxl_vae.safetensors', False)
for filename, url in (
    ('face_yolov8m.pt', 'https://huggingface.co/Bingsu/adetailer/resolve/main/face_yolov8m.pt'),
    ('hand_yolov8s.pt', 'https://huggingface.co/Bingsu/adetailer/resolve/main/hand_yolov8s.pt'),
):
    path = model_root / 'ultralytics' / 'bbox' / filename
    if not path.is_file(): sh(f'wget -q -O "{path}" "{url}"', False)

# Face and hand detailers are optional: setup continues if a public add-on is unavailable.
custom_nodes = Path(comfy) / 'custom_nodes'
custom_nodes.mkdir(parents=True, exist_ok=True)
Path('/content/nodes').mkdir(parents=True, exist_ok=True)
for name in ('ComfyUI-Impact-Pack', 'ComfyUI-Impact-Subpack'):
    local = Path('/content/nodes') / name
    dest = custom_nodes / name
    if dest.is_symlink(): dest.unlink()
    elif dest.exists(): shutil.rmtree(dest, ignore_errors=True)
    if clone_repo(f'https://github.com/ltdrdata/{name}', str(local), False):
        req = local / 'requirements.txt'
        if req.is_file(): sh(f'{sys.executable} -m pip install -q -r "{req}"', False)
        os.symlink(local, dest, target_is_directory=True)
    else:
        print('Optional face/hand detailer unavailable; continuing:', name)

font_source = Path(repo_dir) / 'fonts'
font_target = Path(BASE) / 'fonts'
font_target.mkdir(parents=True, exist_ok=True)
if not list(font_source.glob('Mukta-*.ttf')):
    raise RuntimeError('Bundled Hindi fonts nahi mile. ZIP ko dobara upload karein.')
for font in font_source.glob('Mukta-*.ttf'):
    shutil.copy2(font, font_target / font.name)

os.environ.update(APS_BASE=BASE, APS_CHECKPOINT=model_name, APS_TEXT_MODEL=TEXT_MODEL,
                  HF_HOME='/content/hf_cache', APS_HF_CACHE='/content/hf_cache', APS_COMFY_OUT='/content/comfy_out')
Path('/content/comfy_out').mkdir(exist_ok=True)
Path(BASE, 'data').mkdir(parents=True, exist_ok=True)
print('Setup poora ho gaya ✔')


In [ ]:
#@title 4. Tests (name rule + image prompts/framing)
%cd /content/aps
!python -m pytest -q tests
%cd /content


In [ ]:
#@title 5. Start / Restart — current share link ka intezar karein
import subprocess, time, urllib.request, os, re, signal, sys
from pathlib import Path

def endpoint_ok(url):
    try:
        with urllib.request.urlopen(url, timeout=2):
            return True
    except Exception:
        return False

def recent_log(path, lines=80):
    try:
        return "\n".join(Path(path).read_text(errors="replace").splitlines()[-lines:])
    except OSError:
        return "(log abhi nahi bana)"

comfy_url = 'http://127.0.0.1:8188/system_stats'
comfy_log = '/content/comfyui.log'
if endpoint_ok(comfy_url):
    print('ComfyUI pehle se chal raha ✔')
else:
    comfy_file = open(comfy_log, 'a', buffering=1)
    subprocess.Popen([sys.executable, '-u', 'main.py', '--listen', '127.0.0.1', '--port', '8188',
                      '--output-directory', '/content/comfy_out', '--preview-method', 'none'],
                     cwd='/content/ComfyUI', stdout=comfy_file, stderr=comfy_file,
                     start_new_session=True, env={**os.environ, 'PYTHONUNBUFFERED': '1'})
    for attempt in range(150):
        if endpoint_ok(comfy_url):
            print('ComfyUI chal raha ✔')
            break
        if attempt in (39, 79, 119): print('ComfyUI start ho raha hai…')
        time.sleep(2)
    else:
        raise RuntimeError('ComfyUI start nahi hua. Log ki aakhri lines:\n' + recent_log(comfy_log))

app_url = 'http://127.0.0.1:7860/config'
app_log = '/content/aps-app.log'
app_pid_file = Path('/content/aps-app.pid')
if app_pid_file.is_file():
    try:
        old_pid = int(app_pid_file.read_text().strip())
        os.killpg(old_pid, signal.SIGTERM)
        print('Purana app band; naya share link bana rahe hain…')
        for _ in range(20):
            try:
                os.kill(old_pid, 0)
                time.sleep(0.5)
            except ProcessLookupError:
                break
    except (ValueError, ProcessLookupError, PermissionError):
        pass

Path(app_log).write_text('')
app_file = open(app_log, 'a', buffering=1)
app_process = subprocess.Popen([sys.executable, '-u', '-m', 'app.main'], cwd='/content/aps',
                               stdout=app_file, stderr=app_file, start_new_session=True,
                               env={**os.environ, 'PYTHONUNBUFFERED': '1'})
app_pid_file.write_text(str(app_process.pid))
print('App start ho raha hai; public link milne tak cell ko chalne dein…', flush=True)
link_pattern = re.compile(r'https://[A-Za-z0-9.-]+\.gradio\.live')
share_url = None
local_ready = False
for attempt in range(180):
    if app_process.poll() is not None:
        raise RuntimeError('App process band ho gaya. Log ki aakhri lines:\n' + recent_log(app_log))
    local_ready = local_ready or endpoint_ok(app_url)
    log_text = recent_log(app_log, 300)
    urls = link_pattern.findall(log_text)
    if urls:
        share_url = urls[-1]
        break
    if attempt in (14, 29, 59, 89, 119, 149):
        print('Public link abhi ban raha hai…' if local_ready else 'App start ho raha hai…', flush=True)
    time.sleep(2)

if share_url:
    print('AI Post Studio chal raha ✔')
    print('Naya/current app link:', share_url, flush=True)
else:
    status = 'Local app ready hai, lekin Gradio public tunnel link nahi bana.' if local_ready else 'Local app bhi ready nahi hua.'
    raise RuntimeError(status + '\nColab runtime ka internet check karein, phir Start / Restart dobara chalayein. App log:\n' + recent_log(app_log, 120))


**Tips**
- Har category mein images ka default subject **Sirf ladki (Female)** hai; Couple aur Male options ab bhi hain.
- Chhoti bindi allowed hai; image prompts maang mein sindoor / vermilion ko mana karte hain. Model ke output ko phir bhi check karein.
- **Reference se kya follow karein** aur **Kin cheezon se bachein** settings mein apne visual prompts/notes save karein; yeh naye images aur biodata portraits par apply honge.
- Pehle se bani image se biodata: Images gallery mein photo par click karein, Biodata tab kholein, phir **Selected photo se biodata banao** dabayein. Details random hongi, photo wahi rahegi.
- Biodata mein female aur Half photo default hain; gender aur photo options badal sakte hain.
- **Bold** polished jewel-tone outfits ke liye hai; **Mix** default look hai.
- Pehli run mein ComfyUI aur ek bada photo model download hota hai. Google Drive aur Colab disk mein jagah aur waqt rakhein; rukne par Setup dobara chalayein, partial download resume hoga.
- Face/hand detailer optional hai. Face, kapde, aankhen ya ungliyan har baar perfect hone ki guarantee nahi hoti; final image dekh kar approve karein.
- Full photo biodata mein blur-fill nahi, saaf plain background hai; Half photo chest-up framing ko aur full head ko card par upar se fit kiya gaya hai.
- Free Colab runtime band ho sakta hai; saved images and data Drive mein rehte hain. Band ho to Setup (zaroorat ho to) aur Start / Restart chalakar naya link kholen.
